# 01 · Stage 1 check: collect one round, then check the trial archive

ส่วนที่ 1 ดึงหนึ่งรอบในเครื่อง (ใช้ `force` จึงทำงานแม้สวิตช์ปิดอยู่) · ส่วนที่ 2 อ่านคลังจริงบน Google Drive แล้วสรุปรายสถานี
เรียกใช้ `cpmosaic` เท่านั้น ไม่มีโค้ดกระบวนการใน notebook นี้

In [ ]:
import os, sys, subprocess, pathlib

REPO_URL = "https://github.com/jamorn12/chaophraya-rain.git"   # ใช้เฉพาะบน Colab
IN_COLAB = "google.colab" in sys.modules

def find_root(start=pathlib.Path.cwd()):
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists() and (p / "src" / "cpmosaic").exists():
            return p
    return None

ROOT = find_root()
if ROOT is None and IN_COLAB:
    subprocess.run(["git", "clone", "-q", REPO_URL, "/content/chaophraya-rain"], check=True)
    ROOT = pathlib.Path("/content/chaophraya-rain")
assert ROOT is not None, "run this notebook from inside the repo, or set REPO_URL on Colab"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT)], check=True)
sys.path.insert(0, str(ROOT / "src"))
os.environ["CPMOSAIC_ROOT"] = str(ROOT)
print("repo:", ROOT)

In [ ]:
from cpmosaic.config import radars, load_yaml
from cpmosaic.s1_collect.fetch import Collector
from cpmosaic.s1_collect import inventory

cfg = load_yaml("collect.yaml")
DATA = ROOT / cfg.get("data_root", "data")
c = Collector(root=DATA, user_agent=cfg["user_agent"], timeout_s=cfg["timeout_s"],
              retries=cfg["retries"], backoff_s=cfg["backoff_s"], min_bytes=cfg["min_bytes"])
results = c.run(radars(), repeat=1)          # runs regardless of the switch; also writes the inventory CSV
for r in results:
    print(f"{r.code:4s} {r.status:9s} {r.bytes/1e6:6.2f} MB  {r.width}x{r.height}  {r.error}")

In [ ]:
inv = inventory.load(DATA)
inventory.daily_summary(inv)   # new images per day per radar (96 expected for a full day)

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

new = [r for r in results if r.status == "new"]
fig, axes = plt.subplots(1, len(new), figsize=(3 * len(new), 3.4), squeeze=False)
for ax, r in zip(axes[0], new):
    ax.imshow(Image.open(DATA / r.path)); ax.set_title(r.code); ax.axis("off")
plt.tight_layout(); plt.show()

## 2 · Trial check from the Drive archive (Colab)

ภาพจากทุกสถานีเข้าคลังไหม ได้ภาพใหม่ราว 96 ภาพต่อวันไหม มีช่องว่างนานแค่ไหน

In [ ]:
ARCHIVE = "/content/drive/MyDrive/ChaoPhraya_Radar_Archive"   # same folder as ARCHIVE_REMOTE
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
trial = inventory.load(ARCHIVE)
print(f"{len(trial)} requests, {trial.fetch_utc.min()} to {trial.fetch_utc.max()}")
inventory.trial_report(trial)

In [ ]:
inventory.daily_summary(trial)   # new images per radar per UTC day (96 expected)